# Gộp ứng viên Round 1

Notebook này chỉ đọc câu hỏi có nhãn difficulty/secondary_topics và bảy file trong data/results, rồi gộp các văn bản và Điều trùng nhau theo từng câu hỏi. Nó không gọi LLM hoặc API.

Kết quả được ghi vào data/results/round2_candidate_pool.json. Mỗi câu có id, question, topic, difficulty, secondary_topics, relevant_docs và relevant_articles để đưa cho judge. candidate_support giữ số model và tên file đã đề xuất để kiểm tra độc lập. Khi dán sang ChatGPT, chỉ gửi câu hỏi, topic, difficulty, secondary_topics và hai candidate lists; bỏ metadata phiếu để tránh thiên lệch đồng thuận.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'test' / 'questions_with_difficulty.json').is_file():
    raise FileNotFoundError(f'Run this notebook from the repository root or notebooks folder: {ROOT}')
sys.path.insert(0, str(ROOT))
from notebooks import round2_judge_pipeline as pipeline

QUESTIONS_PATH = ROOT / 'data' / 'test' / 'questions_with_difficulty.json'
RESULT_NAMES = (
    'vilegal_gemini-2.5-flash.json',
    'vilegal_gemini-3.1-flash-lite.json',
    'vilegal_gemini-3.5-flash-lite.json',
    'vilegal_gpt-5.6-luna.json',
    'vilegal_gpt-6-luna.json',
    'vilegal_gpt5-4-mini.json',
    'vilegal_qwen3-235b-a22b-instruct-2507.json',
)
RESULT_PATHS = [ROOT / 'data' / 'results' / name for name in RESULT_NAMES]
OUTPUT_PATH = ROOT / 'data' / 'results' / 'round2_candidate_pool.json'

missing = [str(path) for path in [QUESTIONS_PATH, *RESULT_PATHS] if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing input files: {missing}')

questions = pipeline.load_questions(QUESTIONS_PATH)
question_ids = set(questions)
article_pool, input_review = pipeline.load_candidates(RESULT_PATHS, question_ids)

doc_support = {qid: {} for qid in question_ids}
for path in RESULT_PATHS:
    rows = json.loads(path.read_text(encoding='utf-8-sig'))
    for row in rows:
        if not isinstance(row, dict) or type(row.get('id')) is not int:
            continue
        qid = row['id']
        if qid not in question_ids:
            continue
        model_output = row.get('model_output')
        docs = model_output.get('relevant_docs') if isinstance(model_output, dict) else None
        if not isinstance(docs, list):
            continue
        for doc in docs:
            if isinstance(doc, str) and doc.strip():
                law_id = pipeline.normalize_law_id(doc)
                doc_support[qid].setdefault(law_id, set()).add(path.stem)

records = []
for qid in sorted(questions):
    articles = list(article_pool.get(qid, {}).values())
    articles.sort(key=lambda item: (item['law_id'], item['article']))
    docs = sorted(doc_support[qid])
    reasons = list(dict.fromkeys(input_review.get(qid, [])))
    if not docs and not articles:
        reasons.append('no_candidates_from_round1')
    records.append({
        'id': qid,
        'question': questions[qid]['question'],
        'topic': questions[qid].get('topic'),
        'difficulty': questions[qid].get('difficulty'),
        'secondary_topics': questions[qid].get('secondary_topics', []),
        'relevant_docs': docs,
        'relevant_articles': [
            {'law_id': item['law_id'], 'article': item['article']}
            for item in articles
        ],
        'candidate_support': {
            'relevant_docs': [
                {'law_id': law_id, 'votes': len(doc_support[qid][law_id]),
                 'proposed_by': sorted(doc_support[qid][law_id])}
                for law_id in docs
            ],
            'relevant_articles': [
                {'law_id': item['law_id'], 'article': item['article'],
                 'votes': len(item['proposed_by']), 'proposed_by': sorted(item['proposed_by'])}
                for item in articles
            ],
        },
        'input_review_reasons': reasons,
    })

output = {
    'schema_version': 1,
    'source_files': [QUESTIONS_PATH.name, *[path.name for path in RESULT_PATHS]],
    'question_count': len(records),
    'questions': records,
}
pipeline.write_json_atomic(OUTPUT_PATH, output)

record_by_id = {row['id']: row for row in records}
def show_question(question_id):
    row = record_by_id[question_id]
    payload = {key: row[key] for key in ('id', 'question', 'topic', 'difficulty', 'secondary_topics', 'relevant_docs', 'relevant_articles')}
    print(json.dumps(payload, ensure_ascii=False, indent=2))

print('Câu hỏi:', len(records))
print('File Round 1:', len(RESULT_PATHS))
print('Ứng viên Điều:', sum(len(row['relevant_articles']) for row in records))
print('Câu có lỗi đầu vào cần rà:', sum(bool(row['input_review_reasons']) for row in records))
print('Đã ghi:', OUTPUT_PATH)
print('Ví dụ payload để dán cho judge: show_question(1)')


## In toàn bộ JSON của một câu hỏi

Chạy cell dưới đây rồi nhập ID. Cell in nguyên hàng JSON trong candidate pool để bạn copy, gồm cả candidate_support và input_review_reasons.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
POOL_PATH = ROOT / 'data' / 'results' / 'round2_candidate_pool.json'
pool = json.loads(POOL_PATH.read_text(encoding='utf-8'))

question_id = int(input('Nhập ID câu hỏi: ').strip())
row = next((item for item in pool['questions'] if item['id'] == question_id), None)
if row is None:
    raise KeyError(f'Không có câu hỏi ID {question_id} trong {POOL_PATH}')

json_text = json.dumps(row, ensure_ascii=False, indent=2)
print(json_text)

def copy_with_tkinter(text):
    import tkinter as tk

    clipboard_root = globals().get('_round2_clipboard_root')
    if clipboard_root is None:
        clipboard_root = tk.Tk()
        clipboard_root.withdraw()
        globals()['_round2_clipboard_root'] = clipboard_root
    clipboard_root.clipboard_clear()
    clipboard_root.clipboard_append(text)
    clipboard_root.update()

try:
    if sys.platform == 'win32':
        import base64
        import subprocess

        encoded = base64.b64encode(json_text.encode('utf-8')).decode('ascii')
        powershell = (
            '$encoded = [Console]::In.ReadToEnd(); '
            '$text = [System.Text.Encoding]::UTF8.GetString([Convert]::FromBase64String($encoded)); '
            'Set-Clipboard -Value $text'
        )
        subprocess.run(
            ['powershell.exe', '-NoProfile', '-STA', '-Command', powershell],
            input=encoded, text=True, encoding='ascii', check=True,
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=10,
        )
    else:
        copy_with_tkinter(json_text)
    print('Đã copy toàn bộ JSON vào clipboard.')
except Exception as error:
    try:
        copy_with_tkinter(json_text)
        print('Đã copy toàn bộ JSON vào clipboard qua Tkinter.')
    except Exception as fallback_error:
        print(f'Không copy được clipboard ({type(error).__name__}; {type(fallback_error).__name__}); hãy copy JSON từ output.')
